# Banka Müşteri Churn Tahmini — Ön İşleme

Encoding, türetilmiş değişkenler ve ölçekleme adımları yer almaktadır. Bu notebook'un çalışması için `1_eda.ipynb` çıktısı olan `df` değişkeninin mevcut olması gerekmektedir.

> Bağımsız çalıştırmak için veri yükleme adımı aşağıya eklenmiştir.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

churn = pd.read_csv('Churn.csv')
df = churn.copy()

## 3. Ön İşleme

### 3.1 Encoding

`Geography` ve `Gender` değişkenlerine one-hot encoding uygulanmıştır.

In [ ]:
print(df.columns)
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)
df[['Geography_Germany', 'Geography_Spain', 'Gender_Male']] = df[['Geography_Germany', 'Geography_Spain', 'Gender_Male']].astype(int)
print(df.head(2))

### 3.2 Ülkelere Göre Churn Oranları

In [ ]:
alm_total = df[df['Geography_Germany']==1].shape[0]
alm_churn = df[(df['Geography_Germany']==1) & (df['Exited']==1)].shape[0]
print(f'Almanya churn oranı: {alm_churn/alm_total:.2%}')

isp_total = df[df['Geography_Spain']==1].shape[0]
isp_churn = df[(df['Geography_Spain']==1) & (df['Exited']==1)].shape[0]
print(f'İspanya churn oranı: {isp_churn/isp_total:.2%}')

fr = df[(df['Geography_Spain']==0) & (df['Geography_Germany']==0)]
print(f'Fransa churn oranı: {fr["Exited"].mean():.2%}')

### 3.3 Türetilmiş Değişkenler

Veriden çeşitli oranlar ve gruplandırmalar türetilerek modele ek bilgi sağlanmıştır.

In [ ]:
df1 = df.copy()

# Age gruplandırma: 18-30=1, 30-40=2, 40-50=3, 50-60=4, 60-92=5
df1.loc[(df1['Age']>=18) & (df1['Age']<=30), 'Age'] = 1
df1.loc[(df1['Age']>30) & (df1['Age']<=40), 'Age'] = 2
df1.loc[(df1['Age']>40) & (df1['Age']<=50), 'Age'] = 3
df1.loc[(df1['Age']>50) & (df1['Age']<=60), 'Age'] = 4
df1.loc[(df1['Age']>60) & (df1['Age']<=92), 'Age'] = 5

In [ ]:
# Kredi skoru gruplama (uluslararası tabloya göre)
def kredi_skor_tablosu(row):
    k = row.CreditScore
    if k < 300: return 1
    elif k < 500: return 2
    elif k < 601: return 3
    elif k < 661: return 4
    elif k < 781: return 5
    elif k < 851: return 6
    else: return 7

df1 = df1.assign(credit_score_table=df1.apply(lambda x: kredi_skor_tablosu(x), axis=1))

In [ ]:
# Emeklilik değişkeni (Almanya & İspanya: 65, Fransa: 66)
df1['retired'] = df['Age']
df1.loc[(df1['retired']>=65) & (df1['Geography_Germany']==1), 'retired'] = 1
df1.loc[(df1['retired']>=65) & (df1['Geography_Spain']==1), 'retired'] = 1
df1.loc[(df1['retired']>=66) & (df['Geography_Spain']==0) & (df['Geography_Germany']==0), 'retired'] = 1
df1.loc[(df1['retired']<65) & (df1['Geography_Germany']==1), 'retired'] = 0
df1.loc[(df1['retired']<65) & (df1['Geography_Spain']==1), 'retired'] = 0
df1.loc[(df1['retired']<66) & (df['Geography_Spain']==0) & (df['Geography_Germany']==0), 'retired'] = 0

# Tenure/NumOfProducts
df1['Tenure/NumOfProducts'] = df1['Tenure'] / df1['NumOfProducts']

In [ ]:
# CreditScore < 405 ikili değişken
df1['smallerthan405'] = df['CreditScore']
df1.loc[df1['smallerthan405'] < 405, 'smallerthan405'] = 1
df1.loc[df1['smallerthan405'] > 405, 'smallerthan405'] = 0

# NOP*: NumOfProducts churn riskine göre sıralanmış
# NOP=1: %27 churn | NOP=2: %7 churn | NOP=3: %82 | NOP=4: %100
df1['NOP*'] = df['NumOfProducts']
df1.loc[df1['NOP*'] == 2, 'NOP*'] = 1
df1.loc[df1['NOP*'] == 1, 'NOP*'] = 2
df1.loc[df1['NOP*'] > 2, 'NOP*'] = 3

# Balance0: Balance sıfır mı değil mi?
df1['Balance0'] = df1['Balance']
df1.loc[df1['Balance0'] == 0, 'Balance0'] = 0
df1.loc[df1['Balance0'] != 0, 'Balance0'] = 1

In [ ]:
# Maaş bazlı türetilmiş değişkenler
df1['ES/Age'] = df1['EstimatedSalary'] / (df['Age'] - 17)
df1['Tenure/Age'] = df1['Tenure'] / (df['Age'] - 17)
df1['Balance/ES'] = df1['Balance'] / df1['EstimatedSalary']

# Aylık tahmini maaş
df1['EstimatedSalary'] = df1['EstimatedSalary'] / 12
df1['ES/Tenure'] = df1['EstimatedSalary'] / (df1['Tenure'] + 1)  # +1: sıfıra bölmeyi önler
df1['ES/Score'] = df1['EstimatedSalary'] / df1['credit_score_table']

In [ ]:
# Orijinal değişkenleri çıkar (türetilmişlerle yer değiştirdi)
df1 = df1.drop(['CreditScore', 'Tenure', 'Balance'], axis=1)
df1.head(3)

### 3.4 Ölçekleme

Sayısal değişkenlere Robust Scaler uygulanmıştır. Standart Scaler yerine bu yöntemin tercih edilmesinin nedeni aykırı değerlere karşı daha dayanıklı olmasıdır.

In [ ]:
from sklearn.preprocessing import RobustScaler

df1_num = df1[['Age', 'NumOfProducts', 'EstimatedSalary',
               'credit_score_table', 'Tenure/NumOfProducts', 'NOP*',
               'ES/Age', 'Tenure/Age', 'Balance/ES', 'ES/Tenure', 'ES/Score']]

col = df1_num.columns
x_transformed = pd.DataFrame(RobustScaler().fit(df1_num).transform(df1_num), columns=col)
x_transformed.head()

In [ ]:
# X: Bağımsız değişkenler birleştirme
X = pd.concat([
    x_transformed.loc[:, 'Age':'NumOfProducts'],
    df1.loc[:, 'HasCrCard':'IsActiveMember'],
    x_transformed.loc[:, 'EstimatedSalary'],
    df1.loc[:, 'Geography_Germany':'Gender_Male'],
    x_transformed.loc[:, 'credit_score_table'],
    df1.loc[:, 'retired'],
    x_transformed.loc[:, 'Tenure/NumOfProducts'],
    df1.loc[:, 'smallerthan405'],
    x_transformed.loc[:, 'NOP*'],
    df1.loc[:, 'Balance0'],
    x_transformed.loc[:, 'ES/Age':'ES/Score']
], axis=1)
X.head(2)